In [ ]:
# ============================================================
# 🚀 BASELINE MODELS - SPRINT 3 (VERSIÓN FINAL CORREGIDA)
# ============================================================

# ============================================================
# 🔧 FIX IMPORTS (CRÍTICO)
# ============================================================

import sys
import os

sys.path.append(os.path.abspath(".."))

# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import joblib
from datetime import datetime

from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_validate

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# ============================================================
# 📂 1. CARGA DE DATA
# ============================================================

print("\n📂 Cargando datos...")

X_train = pd.read_csv("../data/processed/X_train_bal.csv")
y_train = pd.read_csv("../data/processed/y_train_bal.csv").values.ravel()

X_test = pd.read_csv("../data/processed/X_test.csv")
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

# ============================================================
# ⚙️ 2. CARGAR PIPELINE
# ============================================================

print("\n⚙️ Cargando pipeline...")

artifact = joblib.load("../models/preprocessor.pkl")

feature_builder = artifact["feature_builder"]
preprocessor = artifact["preprocessor"]

print("✔ Pipeline cargado correctamente")

# ============================================================
# 🤖 3. MODELOS BASELINE
# ============================================================

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42),
    "SVM": SVC(probability=True, random_state=42),
    "KNN": KNeighborsClassifier()
}

print("\n📘 Modelos:")
for m in models:
    print("-", m)

# ============================================================
# 🔁 4. CROSS VALIDATION
# ============================================================

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)  # reducido para estabilidad

scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

# ============================================================
# 🚀 5. ENTRENAMIENTO
# ============================================================

results = []

print("\n🚀 Entrenando modelos...")

for name, model in models.items():
    
    print(f"\n➡️ {name}")
    
    pipe = Pipeline([
        ('feature_builder', feature_builder),
        ('preprocessing', preprocessor),
        ('model', model)
    ])
    
    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )
    
    results.append({
        "Model": name,
        "Accuracy": scores['test_accuracy'].mean(),
        "Precision": scores['test_precision'].mean(),
        "Recall": scores['test_recall'].mean(),
        "F1": scores['test_f1'].mean(),
        "AUC": scores['test_roc_auc'].mean()
    })

# ============================================================
# 📊 6. RESULTADOS
# ============================================================

results_df = pd.DataFrame(results).sort_values(by="F1", ascending=False)

print("\n📊 RESULTADOS:")
print(results_df)

# ============================================================
# 🏆 7. MEJOR MODELO
# ============================================================

best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

print(f"\n🏆 Mejor modelo: {best_model_name}")

final_pipe = Pipeline([
    ('feature_builder', feature_builder),
    ('preprocessing', preprocessor),
    ('model', best_model)
])

final_pipe.fit(X_train, y_train)

# ============================================================
# 💾 8. GUARDADO
# ============================================================

os.makedirs("../models", exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M")

model_path = f"../models/baseline_{best_model_name}_{timestamp}.pkl"

joblib.dump(final_pipe, model_path)

print(f"\n💾 Modelo guardado en: {model_path}")

# ============================================================
# ✅ FIN
# ============================================================

print("\n✅ Notebook listo - sin errores")